# 🔍 Lesson 02 — Oracle 23ai Vector Search
## Student Activity: Search a Wikipedia Article

In this activity you will:
1. **Choose** a Wikipedia article as your dataset
2. **Generate** vector embeddings for each paragraph (the "pre-load phase")
3. **Load** the vectors into Oracle 23ai on freesql.com
4. **Search** the article using natural language — no keyword matching needed

> This is exactly how Netflix, Spotify, and AI assistants find relevant content at scale.

## Step 1 — Install & Import

In [1]:
!pip install sentence-transformers wikipedia-api -q

from sentence_transformers import SentenceTransformer
import wikipediaapi
import re
import textwrap

print("Libraries loaded ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 745.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.9/126.9 kB 4.9 MB/s eta 0:00:00
Libraries loaded ✓


## Step 2 — Choose Your Article

Pick one of the three articles below by setting `ARTICLE_CHOICE` to 1, 2, or 3:

| # | Article | Focus |
|---|---------|-------|
| 1 | Vector database | What they are, how they're built |
| 2 | Word embedding | How words become numbers |
| 3 | Semantic search | How meaning-based search works |

In [8]:
# ── CHANGE THIS: pick 1, 2, or 3 ──
ARTICLE_CHOICE = 1

ARTICLES = {
    1: "Cultural depictions of cats",
    2: "Word embedding",
    3: "Semantic search",
}

ARTICLE_TITLE = ARTICLES[ARTICLE_CHOICE]
print(f"You chose: '{ARTICLE_TITLE}'")

You chose: 'Cultural depictions of cats'


## Step 3 — Fetch & Chunk the Article

In [9]:
wiki = wikipediaapi.Wikipedia(
    user_agent="oracle-vector-search-lesson/1.0",
    language="en"
)

page = wiki.page(ARTICLE_TITLE)
if not page.exists():
    raise ValueError(f"Article '{ARTICLE_TITLE}' not found on Wikipedia")

print(f"✓ Fetched: {page.title}")
print(f"  Length: {len(page.text):,} characters")

# Split into paragraphs, filter short/empty ones
raw_paragraphs = [p.strip() for p in page.text.split('\n') if len(p.strip()) > 120]

# Truncate to 400 chars max per chunk (fits Oracle VARCHAR2(2000) safely)
chunks = []
for i, para in enumerate(raw_paragraphs[:30]):   # cap at 30 chunks for this lesson
    chunk = para[:400]
    # Remove references like [1], [23]
    chunk = re.sub(r'\[\d+\]', '', chunk).strip()
    if len(chunk) > 80:
        chunks.append(chunk)

print(f"  Chunks: {len(chunks)}")
print()
print("Preview of first 3 chunks:")
for i, c in enumerate(chunks[:3]):
    print(f"  [{i+1}] {c[:100]}...")

✓ Fetched: Cultural depictions of cats
  Length: 16,343 characters
  Chunks: 30

Preview of first 3 chunks:
  [1] The cultural depiction of cats and their relationship to humans is old and stretches back over 9,500...
  [2] While the exact history of human interaction with cats is still somewhat vague, a shallow grave site...
  [3] The lineage of today's cats stems from about 4500 BC and came from Europe and Southeast Asia accordi...


## Step 4 — Generate Vector Embeddings

In [10]:
# Load the model (downloads ~90MB on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded ✓")

# Encode all chunks
embeddings = model.encode(chunks, show_progress_bar=True)
print(f"\nGenerated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Generated 30 embeddings, each with 384 dimensions


## Step 5 — Set Up the Database

📋 **Copy the SQL below and run it in freesql.com** (only once — it creates the table)

> freesql.com → SQL Workshop → SQL Commands → paste → Run

In [ ]:
setup_sql = """-- ============================================================
-- Lesson 02 Step 5: Create the doc_chunks table
-- Run this in freesql.com BEFORE loading data
-- ============================================================

-- Drop if it already exists from a previous run
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE doc_chunks';
EXCEPTION WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE doc_chunks (
    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    doc_name     VARCHAR2(200),
    chunk_text   VARCHAR2(2000),
    chunk_vector VECTOR(384, FLOAT32)
);

-- Verify
SELECT table_name FROM user_tables WHERE table_name = 'DOC_CHUNKS';
"""

print(setup_sql)
print("=" * 60)
print("📋 Copy everything above and run it in freesql.com")

## Step 6 — Generate INSERT Statements

Run this cell → copy the output → paste into freesql.com

> **Why the split trick?** Oracle rejects string literals over 4000 chars at parse time.
> A 384-dimension vector is ~4600 chars, so we split it in half using `TO_CLOB() || ...`

In [11]:
def format_vector(embedding):
    """Format embedding as Oracle VECTOR literal, split to avoid ORA-01704."""
    values = ", ".join(f"{v:.8f}" for v in embedding)
    full = f"[{values}]"
    mid = len(full) // 2
    split_pos = full.rindex(',', 0, mid) + 1
    part1 = full[:split_pos]
    part2 = full[split_pos:]
    return f"TO_VECTOR(TO_CLOB('{part1}') || '{part2}', 384, FLOAT32)"

print("-- ============================================================")
print(f"-- Lesson 02: Vector embeddings from '{ARTICLE_TITLE}'")
print("-- Run in freesql.com AFTER Step 5 (table must exist)")
print("-- ============================================================")
print()
for chunk, embedding in zip(chunks, embeddings):
    safe_text = chunk.replace("'", "''")[:390]   # stay under VARCHAR2(2000)
    vec = format_vector(embedding)
    print(f"INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)")
    print(f"VALUES ('{ARTICLE_TITLE[:50]}', '{safe_text}', {vec});")
    print()
print("COMMIT;")
print()
print(f"-- Verify: {len(chunks)} rows expected")
print("SELECT COUNT(*) FROM doc_chunks;")

-- ============================================================
-- Lesson 02: Vector embeddings from 'Cultural depictions of cats'
-- Run in freesql.com AFTER Step 5 (table must exist)
-- ============================================================

INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)
VALUES ('Cultural depictions of cats', 'The cultural depiction of cats and their relationship to humans is old and stretches back over 9,500 years. Cats are featured in the history of many nations, are the subject of legend, and are a favourite subject of artists and writers.', TO_VECTOR(TO_CLOB('[0.08913419, 0.08863831, 0.04830851, 0.03030507, -0.07260856, 0.06484655, -0.08930843, -0.07758662, -0.04661346, -0.03387798, -0.03464170, -0.03164134, -0.00255789, 0.05280005, -0.03390941, -0.02056992, -0.08623004, -0.06193491, 0.02506007, 0.02311028, -0.03318354, -0.06956755, 0.03549873, -0.01774612, -0.03499227, 0.00646599, 0.00828111, -0.06870127, -0.01134755, 0.05612654, -0.04340854, 

## Step 7 — Search Your Article

Change `MY_QUESTION` below, run the cell, copy the SQL, paste it into freesql.com.

In [15]:
MY_QUESTION = "Why did Bowser kidnapped princess Peach?"
TOP_N = 3

q_emb = model.encode([MY_QUESTION])[0]
q_vec = format_vector(q_emb)

print(f"-- Search query: {MY_QUESTION}")
print(f"-- Top {TOP_N} most similar chunks")
print()
print("SELECT")
print("    chunk_id,")
print("    SUBSTR(chunk_text, 1, 100) AS preview,")
print(f"    ROUND(VECTOR_DISTANCE(chunk_vector, {q_vec}, COSINE), 4) AS similarity_score")
print("FROM doc_chunks")
print("ORDER BY similarity_score ASC")
print(f"FETCH FIRST {TOP_N} ROWS ONLY;")

-- Search query: Why did Bowser kidnapped princess Peach?
-- Top 3 most similar chunks

SELECT
    chunk_id,
    SUBSTR(chunk_text, 1, 100) AS preview,
    ROUND(VECTOR_DISTANCE(chunk_vector, TO_VECTOR(TO_CLOB('[-0.03131678, -0.00824910, 0.00968943, 0.07080664, 0.02606557, 0.05489880, 0.07240712, 0.02623424, 0.03909134, -0.01367518, 0.03835486, 0.01486085, 0.03322820, -0.01846017, 0.00495033, 0.03312832, -0.01219090, -0.03511642, -0.06901122, -0.04546347, -0.05533474, -0.01918194, 0.05687186, 0.07541577, -0.06002008, -0.07347246, -0.01383602, 0.05560324, -0.03358507, -0.06484616, -0.02979184, -0.02988638, 0.02763300, 0.06598230, -0.09131593, 0.01242421, 0.09387084, -0.00193994, -0.01511283, -0.05000431, -0.01987430, -0.00877054, -0.01465913, -0.00653317, -0.02972638, 0.00621262, -0.05551556, 0.00118476, 0.05537344, -0.00197449, -0.05563784, 0.03844444, -0.07558867, -0.07558038, 0.03995083, 0.06465033, 0.06264794, -0.05148806, 0.13646765, 0.01982394, 0.00693890, 0.06950653, 0.04184452, 

## 🎯 Activity — Your Turn

Try these three searches. For each one, run Step 7 with a new question, paste the SQL in freesql.com, and write down what you found.

---

**Search 1:** Ask something that IS in the article
> Example: `"How do vector indexes work?"`

What came back? Does it make sense?

---

**Search 2:** Ask something that is RELATED but not a direct quote
> Example: `"fast similarity search at scale"`

Did it find relevant content even though those exact words aren't in the article?

---

**Search 3:** Ask something UNRELATED
> Example: `"how to make pasta"`

What score did you get? Is it high or low? Why?

---

> 💡 **Key insight:** Vector search finds *meaning*, not keywords.
> A score near **0.0** = very similar. A score near **1.0** = very different.